In [ ]:
import os
import pandas as pd
import xarray as xr
import numpy as np
import gsw
import owslib
print(owslib.__version__)

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="argopy")

0.36.0


In [2]:
ruta_base = "/home/adrian/Escritorio/conexion_cigala"
especies = ["sardine", "anchovy", "hake"]

part_datasets = {}
global_limits = {}

print("Procesado de datos biologicos")
for esp in especies:
    fichero_simu = f"Galicia_roms_{esp}_n_h_20250515.nc"
    path_part = os.path.join(ruta_base, fichero_simu)
    
    ds = xr.open_dataset(path_part, engine='netcdf4')
    mask_huevo = ds['hatched'] < 1  
    
    # Filtramos la densidad original: se queda el valor si es huevo, si es larva pasa a NaN
    dens_solo_huevo = ds['egg_to_larvae_dens'].where(mask_huevo)
    ds['frozen_egg_dens'] = dens_solo_huevo.ffill(dim='time')
    
    part_datasets[esp] = ds
    
    # Límites absolutos basados en la nueva variable congelada
    global_limits[esp] = {
        'min': float(np.nanmin(ds['frozen_egg_dens'].values)),
        'max': float(np.nanmax(ds['frozen_egg_dens'].values))
    }
    print(f"{esp.upper()} Rango de densidades: [{global_limits[esp]['min']:.2f} a {global_limits[esp]['max']:.2f}]")

tiempos_simu = part_datasets["sardine"].time.values

# Eclosión del último huevo
ultimo_tiempo_con_huevos = tiempos_simu[0]

for esp in especies:
    ds = part_datasets[esp]
    mask_huevo = ds['hatched'] < 1  # (Usa el mismo criterio de arriba)
    # Buscamos los tiempos donde al menos una partícula siga siendo huevo
    tiempos_activos = ds.time.where(mask_huevo.any(dim='trajectory'), drop=True).values
    if len(tiempos_activos) > 0:
        ultimo_tiempo_con_huevos = max(ultimo_tiempo_con_huevos, tiempos_activos[-1])

print(f"\nEl último huevo de la simulación eclosiona el: {pd.to_datetime(ultimo_tiempo_con_huevos)}")

intervalo_horas = 6  


fechas_a_procesar = [tiempos_simu[0]]
delta_ns = np.timedelta64(intervalo_horas, 'h')
    
for t in tiempos_simu[1:]:
    if t <= ultimo_tiempo_con_huevos:
        if t - fechas_a_procesar[-1] >= delta_ns:
            fechas_a_procesar.append(t)
                

def obtener_url_roms(dt64):
    dt = pd.to_datetime(dt64)
    yyyy = dt.strftime('%Y')
    mm = dt.strftime('%m')
    yyyymmdd = dt.strftime('%Y%m%d')
    dt_next = dt + pd.Timedelta(days=1)
    yyyymmdd_next = dt_next.strftime('%Y%m%d')
    return f'http://193.144.42.111:8080/thredds/dodsC/models/Almacen_Datos/Pendeco/MeteoGalicia_ROMS_raw/{yyyy}/{mm}/{yyyymmdd}/00/ocean_history_a{yyyymmdd}_{yyyymmdd_next}.nc'

Procesado de datos biologicos


/home/adrian/spack/opt/spack/linux-skylake/miniconda3-24.3.0-k6rvhdk2bfkz6czoll5ls2digtuckfxo/lib/python3.12/site-packages/argopy/utils/lists.py:38: UserWarning: An error occurred while loading the ERDDAP data fetcher, it will not be available !
<class 'ValueError'>
Trailing data
  warnings.warn(


SARDINE Rango de densidades: [1024.00 a 1028.56]
ANCHOVY Rango de densidades: [1024.00 a 1030.97]
HAKE Rango de densidades: [1024.00 a 1025.66]

El último huevo de la simulación eclosiona el: 2025-05-19 15:00:00


In [3]:
# Usamos la primera especie de la lista como referencia para detectar la orientación del canal
especie_ref = especies[0]  # Tomará 'sardine'
part_ref = part_datasets[especie_ref]

std_lat = np.nanstd(part_ref['lat'].values)
std_lon = np.nanstd(part_ref['lon'].values)

if std_lat < std_lon:
    section_type = "latitudinal"
    target_coord = np.nanmedian(part_ref['lat'].values)
    print(f"[AUTO-DETECT] Corte LATITUDINAL detectado (Este-Oeste) usando {especie_ref.upper()} como referencia.")
    print(f"              Posición fija: {target_coord:.6f}° Norte")
else:
    section_type = "longitudinal"
    target_coord = np.nanmedian(part_ref['lon'].values)
    print(f"[AUTO-DETECT] Corte LONGITUDINAL detectado (Norte-Sur) usando {especie_ref.upper()} como referencia.")
    print(f"              Posición fija: {target_coord:.6f}° Este/Oeste")

# Mini tolerancia por si las moscas
tol = 1e-4

[AUTO-DETECT] Corte LATITUDINAL detectado (Este-Oeste) usando SARDINE como referencia.
              Posición fija: 42.500000° Norte


In [4]:
def compute_z_r_section(h_sec, zeta_sec, hc, Cs_r, theta_s, theta_b, Vtransform):
    import numpy as np
    N = len(Cs_r)
    pts = h_sec.shape[0]
    z_r_sec = np.zeros((N, pts))
    sc_r = (np.arange(1, N + 1) - N - 0.5) / N
    
    for k in range(N):
        if Vtransform == 1:
            z0 = hc * (sc_r[k] - Cs_r[k]) + Cs_r[k] * h_sec
            z_r_sec[k] = z0 + zeta_sec * (1 + z0 / h_sec)
        elif Vtransform == 2:
            z0 = (hc * sc_r[k] + Cs_r[k] * h_sec) / (hc + h_sec)
            z_r_sec[k] = zeta_sec + (zeta_sec + h_sec) * z0
        z_r_sec[k] = np.minimum(z_r_sec[k], 0)
        z_r_sec[k] = np.maximum(z_r_sec[k], -h_sec)
    return z_r_sec

In [5]:
carpeta_salida = "render_isopicnas"
os.makedirs(carpeta_salida, exist_ok=True)

# CAMBIAR AQUI CUANTAS PLOTEAR
max_particulas = 50  

print(f"Plotting...")

for num_progreso, fecha_target in enumerate(fechas_a_procesar, start=1):
    fecha_str = pd.to_datetime(fecha_target).strftime('%Y%m%d_%H%M')
    print(f"[{num_progreso}/{len(fechas_a_procesar)}] Fecha: {pd.to_datetime(fecha_target)}")
    
    url_roms = obtener_url_roms(fecha_target)
    ds_roms = xr.open_dataset(url_roms)
    
    if section_type == "latitudinal":
        lat_mean = ds_roms['lat_rho'].mean(dim='xi_rho').values
        idx = np.argmin(np.abs(lat_mean - target_coord))
        roms_sec = ds_roms.isel(eta_rho=idx).sel(ocean_time=fecha_target, method="nearest")
        roms_sec = roms_sec.interpolate_na(dim="xi_rho", method="linear")
        coord_horizontal = roms_sec['lon_rho'].values
        coord_label = "Longitud (°E)"
    elif section_type == "longitudinal":
        lon_mean = ds_roms['lon_rho'].mean(dim='eta_rho').values
        idx = np.argmin(np.abs(lon_mean - target_coord))
        roms_sec = ds_roms.isel(xi_rho=idx).sel(ocean_time=fecha_target, method="nearest")
        roms_sec = roms_sec.interpolate_na(dim="eta_rho", method="linear")
        coord_horizontal = roms_sec['lat_rho'].values
        coord_label = "Latitud (°N)"
        
    temp_sec = roms_sec['temp'].values
    salt_sec = roms_sec['salt'].values
    zeta_sec = np.nan_to_num(roms_sec['zeta'].values, nan=0)
    h_sec = roms_sec['h'].values

    # Geometría vertilac
    theta_s = ds_roms['theta_s'].values
    theta_b = ds_roms['theta_b'].values
    hc = ds_roms['hc'].values
    Cs_r = ds_roms['Cs_r'].values
    Vtransform = ds_roms['Vtransform'].values if 'Vtransform' in ds_roms else 2

    z_r_sec = compute_z_r_section(h_sec, zeta_sec, hc, Cs_r, theta_s, theta_b, Vtransform)
    depth_sec = -z_r_sec  
    coord_2d = np.tile(coord_horizontal, (z_r_sec.shape[0], 1))

    # Variables en el roms
    lat0 = np.nanmean(coord_horizontal)
    lon0 = target_coord if section_type == "longitudinal" else np.nanmean(coord_horizontal)
    mask_water = ~np.isnan(salt_sec)
    
    SA_sec = np.full(salt_sec.shape, np.nan)
    CT_sec = np.full(temp_sec.shape, np.nan)
    rho_sec = np.full(temp_sec.shape, np.nan)
    
    pressure_sec = gsw.p_from_z(z_r_sec, lat0)  
    SA_sec[mask_water] = gsw.SA_from_SP(salt_sec[mask_water], pressure_sec[mask_water]/10., lon0, lat0)
    CT_sec[mask_water] = gsw.CT_from_t(SA_sec[mask_water], temp_sec[mask_water], pressure_sec[mask_water]/10.)
    rho_sec[mask_water] = gsw.rho(SA_sec[mask_water], CT_sec[mask_water], pressure_sec[mask_water]/10.)

    # Filtrado aleatorio
    particulas_filtradas = {}
    promedios_desarrollo = {}  
    all_x_presentes = []
    
    for esp in especies:
        ds = part_datasets[esp]
        part_paso = ds.sel(time=fecha_target, method="nearest")
        
        lon_inst = part_paso['lon'].values
        lat_inst = part_paso['lat'].values
        z_inst = -part_paso['z'].values
        dens_inst = part_paso['frozen_egg_dens'].values
        
        stage_inst = part_paso['stage_fraction'].values if 'stage_fraction' in part_paso else np.nan
        
        mask_vivas = ~np.isnan(lon_inst) & ~np.isnan(z_inst) & ~np.isnan(dens_inst)
        
        if section_type == "longitudinal":
            mask_seccion = mask_vivas & (np.abs(lon_inst - target_coord) <= tol)
            x_p = lat_inst[mask_seccion]
        elif section_type == "latitudinal":
            mask_seccion = mask_vivas & (np.abs(lat_inst - target_coord) <= tol)
            x_p = lon_inst[mask_seccion]
            
        y_p = z_inst[mask_seccion]
        c_p = dens_inst[mask_seccion]
        
        if len(y_p) > 0 and not np.all(np.isnan(stage_inst)):
            stage_p = stage_inst[mask_seccion]
            promedios_desarrollo[esp] = np.nanmean(stage_p)
        else:
            promedios_desarrollo[esp] = np.nan  
        
        if len(x_p) > max_particulas:
            indices_aleatorios = np.random.choice(len(x_p), max_particulas, replace=False)
            x_p = x_p[indices_aleatorios]
            y_p = y_p[indices_aleatorios]
            c_p = c_p[indices_aleatorios]
            
        particulas_filtradas[esp] = {'x': x_p, 'y': y_p, 'c': c_p}
        if len(x_p) > 0:
            all_x_presentes.extend(x_p)

    depth_min, depth_limit, num_isopicnas = 0, 100, 80  # <-- DEVUELTO A LA VIDA
    fig, ax = plt.subplots(figsize=(13, 6.8))  
    
    ax.set_axisbelow(True)
    ax.grid(True, linestyle="--", alpha=0.4, zorder=0)
    
    # Isopicnas físicas (zorder=1)
    rho_masked = np.ma.masked_invalid(rho_sec)
    mask_rango_prof = (depth_sec >= depth_min) & (depth_sec <= depth_limit)
    vmin_iso = np.nanmin(rho_masked[mask_rango_prof]) if np.any(mask_rango_prof) else 1024.2
    vmax_iso = np.nanmax(rho_masked[mask_rango_prof]) if np.any(mask_rango_prof) else 1028.5
    levels = np.linspace(vmin_iso, vmax_iso, num_isopicnas)
    norm_iso = plt.Normalize(vmin=levels.min(), vmax=levels.max())
    
    ax.contour(coord_2d, depth_sec, rho_masked, levels=levels, cmap="viridis", linewidths=1.0, zorder=1)
    
    def generar_cmap_variable(base_name):
        base = plt.get_cmap(base_name)
        colores = base(np.linspace(0.2, 1.0, 256))
        colores[:, -1] = np.linspace(0.1, 1.0, 256)
        return LinearSegmentedColormap.from_list(f"Custom_{base_name}", colores)
        
    cmaps_especies = {
        'sardine': generar_cmap_variable('Reds'),
        'anchovy': generar_cmap_variable('Blues'),
        'hake':    generar_cmap_variable('Greens')
    }
    
    # Pintado de partículas (zorder=3)
    for esp, datos in particulas_filtradas.items():
        if len(datos['x']) > 0:
            ax.scatter(
                datos['x'], datos['y'], c=datos['c'],
                cmap=cmaps_especies[esp], s=45, edgecolor="black", linewidth=0.5,
                vmin=global_limits[esp]['min'], vmax=global_limits[esp]['max'],
                zorder=3
            )
            
    if len(all_x_presentes) > 0:
        buffer = 0.03
        x_min_plot = max(np.min(all_x_presentes) - buffer, np.nanmin(coord_horizontal))
        x_max_plot = min(np.max(all_x_presentes) + buffer, np.nanmax(coord_horizontal))
    else:
        x_min_plot, x_max_plot = np.nanmin(coord_horizontal), np.nanmax(coord_horizontal)
        
    ax.set_xlim(x_min_plot, x_max_plot)
    ax.set_ylim(depth_limit, depth_min - 2)
    
    ax.set_title(f"Vertical section ({section_type.upper()}) | Egg Stage (Max: {max_particulas} part.)\nSection coord: {target_coord:.4f}° | Date: {pd.to_datetime(fecha_target).strftime('%Y-%d-%m %H:%M')}", fontsize=11, fontweight='bold')
    ax.set_xlabel(coord_label)
    ax.set_ylabel("Depth (m)")

    plt.subplots_adjust(left=0.07, right=0.95, top=0.88, bottom=0.28)

    cax_iso     = fig.add_axes([0.07, 0.09, 0.18, 0.025])
    cax_sardine = fig.add_axes([0.30, 0.09, 0.18, 0.025])
    cax_anchovy = fig.add_axes([0.53, 0.09, 0.18, 0.025])
    cax_hake    = fig.add_axes([0.76, 0.09, 0.18, 0.025])

    # Densidades (isopicnas)
    sm_iso = cm.ScalarMappable(norm=norm_iso, cmap="viridis")
    cbar_iso = fig.colorbar(sm_iso, cax=cax_iso, orientation='horizontal')
    cbar_iso.set_label("Isopycnic (kg/m³)", fontweight='bold', fontsize=9)
    cbar_iso.ax.tick_params(labelsize=8)
    cbar_iso.ax.xaxis.set_major_locator(MaxNLocator(nbins=3, prune='both'))

    # Sardina
    st_sardine = promedios_desarrollo['sardine']
    txt_sardine = f"SARDINE (kg/m³)\nAvg Stage Fraction: {st_sardine:.2f}" if not np.isnan(st_sardine) else "SARDINE (kg/m³)\nAvg Stage: N/A"
    norm_sardine = plt.Normalize(vmin=global_limits['sardine']['min'], vmax=global_limits['sardine']['max'])
    cbar_sardine = fig.colorbar(cm.ScalarMappable(norm=norm_sardine, cmap=cmaps_especies['sardine']), cax=cax_sardine, orientation='horizontal')
    cbar_sardine.set_label(txt_sardine, color="darkred", fontweight='bold', fontsize=9)
    cbar_sardine.ax.tick_params(labelsize=8)
    cbar_sardine.ax.xaxis.set_major_locator(MaxNLocator(nbins=3))

    # Anchoa
    st_anchovy = promedios_desarrollo['anchovy']
    txt_anchovy = f"ANCHOVY (kg/m³)\nAvg Stage Fraction: {st_anchovy:.2f}" if not np.isnan(st_anchovy) else "ANCHOVY (kg/m³)\nAvg Stage: N/A"
    norm_anchovy = plt.Normalize(vmin=global_limits['anchovy']['min'], vmax=global_limits['anchovy']['max'])
    cbar_anchovy = fig.colorbar(cm.ScalarMappable(norm=norm_anchovy, cmap=cmaps_especies['anchovy']), cax=cax_anchovy, orientation='horizontal')
    cbar_anchovy.set_label(txt_anchovy, color="darkblue", fontweight='bold', fontsize=9)
    cbar_anchovy.ax.tick_params(labelsize=8)
    cbar_anchovy.ax.xaxis.set_major_locator(MaxNLocator(nbins=3))

    # Merluza
    st_hake = promedios_desarrollo['hake']
    txt_hake = f"HAKE (kg/m³)\nAvg Stage Fraction: {st_hake:.2f}" if not np.isnan(st_hake) else "HAKE (kg/m³)\nAvg Stage: N/A"
    norm_hake = plt.Normalize(vmin=global_limits['hake']['min'], vmax=global_limits['hake']['max'])
    cbar_hake = fig.colorbar(cm.ScalarMappable(norm=norm_hake, cmap=cmaps_especies['hake']), cax=cax_hake, orientation='horizontal')
    cbar_hake.set_label(txt_hake, color="darkgreen", fontweight='bold', fontsize=9)
    cbar_hake.ax.tick_params(labelsize=8)
    cbar_hake.ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
    
    nombre_archivo = f"corte_isopicnas_{fecha_str}.png"
    ruta_guardado = os.path.join(carpeta_salida, nombre_archivo)
    plt.savefig(ruta_guardado, dpi=150)
    plt.close(fig)  
    print(f"Guardado en: {nombre_archivo}\n")

Plotting...
[1/19] Fecha: 2025-05-15 00:00:00
Guardado en: corte_isopicnas_20250515_0000.png

[2/19] Fecha: 2025-05-15 06:00:00
Guardado en: corte_isopicnas_20250515_0600.png

[3/19] Fecha: 2025-05-15 12:00:00
Guardado en: corte_isopicnas_20250515_1200.png

[4/19] Fecha: 2025-05-15 18:00:00
Guardado en: corte_isopicnas_20250515_1800.png

[5/19] Fecha: 2025-05-16 00:00:00
Guardado en: corte_isopicnas_20250516_0000.png

[6/19] Fecha: 2025-05-16 06:00:00
Guardado en: corte_isopicnas_20250516_0600.png

[7/19] Fecha: 2025-05-16 12:00:00
Guardado en: corte_isopicnas_20250516_1200.png

[8/19] Fecha: 2025-05-16 18:00:00
Guardado en: corte_isopicnas_20250516_1800.png

[9/19] Fecha: 2025-05-17 00:00:00
Guardado en: corte_isopicnas_20250517_0000.png

[10/19] Fecha: 2025-05-17 06:00:00
Guardado en: corte_isopicnas_20250517_0600.png

[11/19] Fecha: 2025-05-17 12:00:00
Guardado en: corte_isopicnas_20250517_1200.png

[12/19] Fecha: 2025-05-17 18:00:00
Guardado en: corte_isopicnas_20250517_1800.png

[